# BnSentMix — Model 3: Class-Balanced Focal XLM-R

**Hypothesis H3:** Model 2's remaining mixed-class recall deficit is partly caused by optimization bias toward frequent/easy classes. Replacing ordinary cross-entropy with a predeclared class-balanced focal loss should improve mean validation mixed-F1 while keeping mean validation macro-F1 within 0.005 of Model 2.

This is a **loss-only ablation**. The dataset, split, XLM-R backbone, classification head, seeds, maximum length, optimizer, learning-rate schedule, epoch budget, early stopping rule, and selection metric remain fixed. The final test stays sealed.

Model 2 validation result: macro-F1 `0.7182 ± 0.0041`, mixed-F1 `0.6282 ± 0.0162`, accuracy `0.7362 ± 0.0017`. For reference seed 42, mixed precision/recall/F1 were `0.7195 / 0.5464 / 0.6211`, so recall is the clearest remaining weakness.

## Predeclared intervention and decision rule

For a training example of class $y$, the loss is

$$L=-\tilde{w}_y(1-p_y)^\gamma\log(p_y), \qquad w_y=\frac{1-\beta}{1-\beta^{n_y}}$$

where $n_y$ is the **training-only** class count and the weights are normalized to mean 1. We fix `beta=0.9999` and `gamma=2.0` before training; there is no validation tuning. Effective-number weighting follows [Cui et al. (CVPR 2019)](https://openaccess.thecvf.com/content_CVPR_2019/html/Cui_Class-Balanced_Loss_Based_on_Effective_Number_of_Samples_CVPR_2019_paper.html), and the focusing term follows [Lin et al. (ICCV 2017)](https://openaccess.thecvf.com/content_ICCV_2017/html/Lin_Focal_Loss_for_ICCV_2017_paper.html). This notebook uses the mutually exclusive, four-class softmax form.

**Support rule:** H3 is supported on validation only if (1) mean mixed-F1 is greater than `0.6282`, and (2) mean macro-F1 is at least `0.7132` (Model 2 minus the predeclared 0.005 margin). With three seeds, deltas are descriptive; do not claim statistical significance.

## Fixed parameters

| Item | Fixed value |
|---|---|
| Dataset | `/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv` |
| Locked split | Group-disjoint 70/15/15, split seed 2026 |
| Backbone/head | `FacebookAI/xlm-roberta-base`, four-way softmax |
| Only Model 3 change | Class-balanced focal loss (`beta=0.9999`, `gamma=2.0`) |
| Maximum length | 128 tokens |
| Per-device batch / accumulation | 16 / 2 |
| Learning rate / weight decay | 2e-5 / 0.01 |
| Epoch budget / patience | 5 / 2 |
| Run seeds | 13, 42, 77 |
| Selection metric | Validation macro-F1 |
| Test policy | Sealed until Models 1–6 are frozen |

> **Checkpoint note:** the uploaded Model 2 log showed a `LayerNorm.gamma/beta` key mismatch during automatic best-checkpoint reload. Model 3 therefore stores the exact in-memory `state_dict` and reloads it with `strict=True`. Before publication, rerun Model 2 with the same safe checkpoint mechanism so the loss is the only difference.

In [ ]:
# Core imports and all parameters are declared before training.
from pathlib import Path
import gc
import hashlib
import inspect
import json
import math
import re
import unicodedata

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import Dataset
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_fscore_support
)
from sklearn.model_selection import StratifiedGroupKFold
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    DataCollatorWithPadding, Trainer, TrainerCallback,
    TrainingArguments, set_seed
)

DATA_PATH = Path('/kaggle/input/datasets/olidopu/bnsentmix/dataset.csv')
PROTOCOL_DIR = Path('/kaggle/working/bnsentmix_protocol_2026')
MODEL_DIR = Path('/kaggle/working/bnsentmix_model3_cb_focal')
PROTOCOL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Fingerprints copied from the locked Model 1/2 protocol.
EXPECTED_RAW_SHA256 = '148f23eb3dc40c1012a973efec920eaccc39700a74e5bcfb56806b0bf389029d'
EXPECTED_SPLIT_SHA256 = '7be81e52030967319d43ddc1e1feaf9817f9c3011a9b914265c34b5ce0ab8ec5'
EXPECTED_SPLIT_COUNTS = {'train': 13862, 'validation': 2975, 'test': 2973}
SPLIT_SEED = 2026

# Labels were already confirmed; no label audit is repeated.
LABELS = [0, 1, 2, 3]
LABEL_NAMES = {0: 'positive', 1: 'negative', 2: 'neutral', 3: 'mixed'}
MIXED_LABEL = 3

# Model 2 controls.
MODEL_NAME = 'FacebookAI/xlm-roberta-base'
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.10
MAX_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
RUN_SEEDS = [13, 42, 77]
REFERENCE_SEED = 42
OPEN_FINAL_TEST = False  # Keep False until Models 1–6 are frozen.

# Model 3 intervention and predeclared acceptance margin.
CB_BETA = 0.9999
FOCAL_GAMMA = 2.0
MACRO_NONINFERIORITY_MARGIN = 0.005
MODEL2_MEAN_MACRO_F1 = 0.7182
MODEL2_MEAN_MIXED_F1 = 0.6282

USE_FP16 = torch.cuda.is_available()
GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('GPU count:', GPU_COUNT)
print('Device 0:', torch.cuda.get_device_name(0) if GPU_COUNT else 'CPU')
print('Final test open:', OPEN_FINAL_TEST)

In [ ]:
# Reconstruct Model 1's split exactly and verify both fingerprints.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def normalized_group_key(text):
    text = unicodedata.normalize('NFKC', str(text)).lower()
    text = re.sub(r'[\u200b-\u200d\ufeff]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

raw = pd.read_csv(DATA_PATH)
assert sha256_file(DATA_PATH) == EXPECTED_RAW_SHA256, 'Raw dataset differs from Models 1–2.'

lookup = {column.strip().lower(): column for column in raw.columns}
text_col = next(lookup[name] for name in ['sentence', 'text', 'comment'] if name in lookup)
label_col = next(lookup[name] for name in ['label', 'sentiment', 'class'] if name in lookup)
df = raw[[text_col, label_col]].rename(columns={text_col: 'text', label_col: 'label'}).copy()
df.insert(0, 'source_row_id', np.arange(len(df)))
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].ne('')].copy()
df['group_id'] = df['text'].map(normalized_group_key)
# Remove same-text/same-label copies; cross-label identical texts remain in one group.
df = df.drop_duplicates(['group_id', 'label'], keep='first').reset_index(drop=True)

splitter = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SPLIT_SEED)
fold_ids = np.full(len(df), -1, dtype=int)
for fold_id, (_, held_out) in enumerate(
    splitter.split(df['text'], df['label'], groups=df['group_id'])
):
    fold_ids[held_out] = fold_id
df['split'] = np.where(fold_ids < 14, 'train', np.where(fold_ids < 17, 'validation', 'test'))

assert df.groupby('group_id')['split'].nunique().max() == 1
assert df['split'].value_counts().to_dict() == EXPECTED_SPLIT_COUNTS
split_path = PROTOCOL_DIR / 'bnsentmix_locked_split.csv'
df.to_csv(split_path, index=False)
assert sha256_file(split_path) == EXPECTED_SPLIT_SHA256, 'Split differs from Models 1–2.'

train_df = df[df['split'].eq('train')].copy()
val_df = df[df['split'].eq('validation')].copy()
test_df = df[df['split'].eq('test')].copy()
display(pd.crosstab(df['split'], df['label']).rename(columns=LABEL_NAMES))
print('Verified split SHA-256:', EXPECTED_SPLIT_SHA256)

In [ ]:
# Compute effective-number weights from the training partition only.
train_counts = train_df['label'].value_counts().reindex(LABELS).astype(np.float64)
effective_numbers = 1.0 - np.power(CB_BETA, train_counts.to_numpy())
raw_weights = (1.0 - CB_BETA) / effective_numbers
normalized_weights = raw_weights / raw_weights.sum() * len(LABELS)
CLASS_WEIGHTS = torch.tensor(normalized_weights, dtype=torch.float32)

weight_table = pd.DataFrame({
    'label': LABELS,
    'class_name': [LABEL_NAMES[label] for label in LABELS],
    'train_count': train_counts.to_numpy(dtype=int),
    'normalized_weight': normalized_weights,
})
display(weight_table.round(4))
assert np.isclose(CLASS_WEIGHTS.mean().item(), 1.0, atol=1e-6)
print('Only training labels were used to calculate these fixed weights.')

In [ ]:
# Tokenization and datasets are identical to Model 2.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

class SentimentDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.labels = frame['label'].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = tokenizer(
            self.texts[index], truncation=True, max_length=MAX_LENGTH
        )
        encoded['labels'] = self.labels[index]
        return encoded

train_dataset = SentimentDataset(train_df)
val_dataset = SentimentDataset(val_df)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, pad_to_multiple_of=8 if USE_FP16 else None
)

# Trainer multiplies per-device batch size by the visible GPU count.
VISIBLE_DEVICES = max(GPU_COUNT, 1)
GLOBAL_EFFECTIVE_BATCH_SIZE = (
    TRAIN_BATCH_SIZE * VISIBLE_DEVICES * GRADIENT_ACCUMULATION_STEPS
)
BATCHES_PER_EPOCH = math.ceil(len(train_dataset) / (TRAIN_BATCH_SIZE * VISIBLE_DEVICES))
UPDATES_PER_EPOCH = math.ceil(BATCHES_PER_EPOCH / GRADIENT_ACCUMULATION_STEPS)
TOTAL_UPDATE_STEPS = UPDATES_PER_EPOCH * MAX_EPOCHS
WARMUP_STEPS = math.ceil(TOTAL_UPDATE_STEPS * WARMUP_FRACTION)

print('Training / validation:', len(train_dataset), '/', len(val_dataset))
print('Detected global effective batch size:', GLOBAL_EFFECTIVE_BATCH_SIZE)
print('Maximum optimizer steps / warmup steps:', TOTAL_UPDATE_STEPS, '/', WARMUP_STEPS)

In [ ]:
# Metrics match Model 2; mixed precision and recall are added for diagnosis.
def calculate_metrics(labels, predictions):
    labels = np.asarray(labels)
    predictions = np.asarray(predictions)
    mixed_precision, mixed_recall, mixed_f1, _ = precision_recall_fscore_support(
        labels == MIXED_LABEL, predictions == MIXED_LABEL,
        average='binary', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(labels, predictions)),
        'macro_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='macro', zero_division=0
        )),
        'weighted_f1': float(f1_score(
            labels, predictions, labels=LABELS, average='weighted', zero_division=0
        )),
        'mixed_precision': float(mixed_precision),
        'mixed_recall': float(mixed_recall),
        'mixed_f1': float(mixed_f1),
    }

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    if isinstance(logits, tuple):
        logits = logits[0]
    return calculate_metrics(labels, np.argmax(logits, axis=1))

ID2LABEL = {label: name for label, name in LABEL_NAMES.items()}
LABEL2ID = {name: label for label, name in LABEL_NAMES.items()}

In [ ]:
# Only this loss differs from Model 2's ordinary cross-entropy.
class ClassBalancedFocalTrainer(Trainer):
    def __init__(self, *args, class_weights, focal_gamma, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.detach().clone()
        self.focal_gamma = float(focal_gamma)

    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits

        # CE=-log(p_y); exp(-CE)=p_y for the correct class.
        per_example_ce = F.cross_entropy(logits, labels, reduction='none')
        p_t = torch.exp(-per_example_ce)
        weights = self.class_weights.to(
            device=logits.device, dtype=logits.dtype
        )[labels]
        loss = (weights * (1.0 - p_t).pow(self.focal_gamma) * per_example_ce).mean()
        return (loss, outputs) if return_outputs else loss

# Save the best in-memory parameters directly; this avoids the gamma/beta
# serialization mismatch observed in the uploaded Model 2 run.
class ExactBestStateCallback(TrainerCallback):
    def __init__(self, state_path, patience):
        self.state_path = Path(state_path)
        self.patience = int(patience)
        self.best_score = -np.inf
        self.best_epoch = None
        self.best_loss = None
        self.bad_epochs = 0

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        score = metrics.get('eval_macro_f1') if metrics else None
        if score is None:
            return control

        if score > self.best_score:
            self.best_score = float(score)
            self.best_epoch = float(state.epoch)
            self.best_loss = float(metrics['eval_loss'])
            self.bad_epochs = 0
            unwrapped_model = model.module if hasattr(model, 'module') else model
            cpu_state = {
                key: value.detach().cpu()
                for key, value in unwrapped_model.state_dict().items()
            }
            torch.save({'model_state_dict': cpu_state}, self.state_path)
        else:
            self.bad_epochs += 1
            if self.bad_epochs >= self.patience:
                control.should_training_stop = True
        return control

def load_exact_state(path):
    # weights_only exists in recent PyTorch; this remains compatible with older Kaggle images.
    load_kwargs = {'map_location': 'cpu'}
    if 'weights_only' in inspect.signature(torch.load).parameters:
        load_kwargs['weights_only'] = True
    return torch.load(path, **load_kwargs)['model_state_dict']

In [ ]:
# Version-compatible arguments; warmup_steps replaces deprecated warmup_ratio.
def make_training_arguments(seed, output_dir):
    arguments = dict(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=MAX_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type='linear',
        max_grad_norm=1.0,
        save_strategy='no',
        logging_strategy='epoch',
        load_best_model_at_end=False,
        seed=seed,
        data_seed=seed,
        fp16=USE_FP16,
        dataloader_num_workers=2,
        report_to=[],
    )
    argument_names = inspect.signature(TrainingArguments.__init__).parameters
    arguments['eval_strategy' if 'eval_strategy' in argument_names else 'evaluation_strategy'] = 'epoch'
    return TrainingArguments(**arguments)

def build_fresh_model():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(LABELS),
        id2label=ID2LABEL, label2id=LABEL2ID
    )

In [ ]:
# Train one independent seed and evaluate its validation-selected state.
def train_one_seed(seed):
    seed_dir = MODEL_DIR / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    best_state_path = seed_dir / 'best_state.pt'
    set_seed(seed)

    model = build_fresh_model()
    best_state_callback = ExactBestStateCallback(
        best_state_path, EARLY_STOPPING_PATIENCE
    )
    trainer = ClassBalancedFocalTrainer(
        model=model,
        args=make_training_arguments(seed, seed_dir),
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[best_state_callback],
        class_weights=CLASS_WEIGHTS,
        focal_gamma=FOCAL_GAMMA,
    )
    trainer.train()

    if not best_state_path.exists():
        raise RuntimeError('No validation-selected state was saved.')
    # strict=True verifies that every parameter is restored exactly once.
    model.load_state_dict(load_exact_state(best_state_path), strict=True)
    prediction_output = trainer.predict(val_dataset)
    predictions = np.argmax(prediction_output.predictions, axis=1)
    metrics = calculate_metrics(val_df['label'].to_numpy(), predictions)
    metrics.update({
        'seed': seed,
        'best_epoch': best_state_callback.best_epoch,
        'validation_loss': best_state_callback.best_loss,
        'best_state_path': str(best_state_path),
    })

    prediction_frame = val_df[['source_row_id', 'group_id', 'label']].copy()
    prediction_frame['prediction'] = predictions
    prediction_frame.to_csv(seed_dir / 'validation_predictions.csv', index=False)
    pd.DataFrame(trainer.state.log_history).to_csv(
        seed_dir / 'training_log.csv', index=False
    )

    del trainer, model, prediction_output
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return metrics, predictions

validation_rows = []
validation_predictions = {}
for run_seed in RUN_SEEDS:
    print(f'\n===== Training seed {run_seed} =====')
    seed_metrics, seed_predictions = train_one_seed(run_seed)
    validation_rows.append(seed_metrics)
    validation_predictions[run_seed] = seed_predictions

validation_results = pd.DataFrame(validation_rows).sort_values('seed').reset_index(drop=True)
metric_columns = [
    'accuracy', 'macro_f1', 'weighted_f1',
    'mixed_precision', 'mixed_recall', 'mixed_f1'
]
validation_summary = validation_results[metric_columns].agg(['mean', 'std']).T
validation_results.to_csv(MODEL_DIR / 'validation_seed_results.csv', index=False)
validation_summary.to_csv(MODEL_DIR / 'validation_mean_std.csv')
tokenizer.save_pretrained(MODEL_DIR / 'tokenizer')
display(validation_results.drop(columns='best_state_path').round(4))
display(validation_summary.round(4))

In [ ]:
# Paired, same-seed comparison against the uploaded Model 2 results.
model2_by_seed = pd.DataFrame({
    'seed': [13, 42, 77],
    'model2_accuracy': [0.7351, 0.7355, 0.7382],
    'model2_macro_f1': [0.7158, 0.7159, 0.7229],
    'model2_weighted_f1': [0.7348, 0.7342, 0.7372],
    'model2_mixed_f1': [0.6168, 0.6211, 0.6467],
})
paired = model2_by_seed.merge(validation_results, on='seed', validate='one_to_one')
for metric in ['accuracy', 'macro_f1', 'weighted_f1', 'mixed_f1']:
    paired[f'delta_{metric}'] = paired[metric] - paired[f'model2_{metric}']

comparison_columns = [
    'seed', 'model2_macro_f1', 'macro_f1', 'delta_macro_f1',
    'model2_mixed_f1', 'mixed_f1', 'delta_mixed_f1'
]
display(paired[comparison_columns].round(4))
paired.to_csv(MODEL_DIR / 'model2_vs_model3_paired_validation.csv', index=False)

mean_macro = float(validation_results['macro_f1'].mean())
mean_mixed = float(validation_results['mixed_f1'].mean())
delta_macro = mean_macro - MODEL2_MEAN_MACRO_F1
delta_mixed = mean_mixed - MODEL2_MEAN_MIXED_F1
h3_supported = (
    delta_mixed > 0 and
    delta_macro >= -MACRO_NONINFERIORITY_MARGIN
)
decision = pd.DataFrame([{
    'model3_mean_macro_f1': mean_macro,
    'delta_macro_vs_model2': delta_macro,
    'model3_mean_mixed_f1': mean_mixed,
    'delta_mixed_vs_model2': delta_mixed,
    'predeclared_macro_margin': MACRO_NONINFERIORITY_MARGIN,
    'H3_supported_on_validation': h3_supported,
}])
decision.to_csv(MODEL_DIR / 'hypothesis_H3_decision.csv', index=False)
display(decision.round(4))
print(
    'Interpretation:',
    'H3 supported on validation.' if h3_supported else
    'H3 not supported under the predeclared validation rule.'
)

In [ ]:
# Per-class diagnosis for the predeclared reference seed.
reference_prediction = validation_predictions[REFERENCE_SEED]
reference_report = classification_report(
    val_df['label'], reference_prediction, labels=LABELS,
    target_names=[LABEL_NAMES[label] for label in LABELS],
    output_dict=True, zero_division=0
)
display(pd.DataFrame(reference_report).T.round(4))
display(pd.DataFrame(
    confusion_matrix(val_df['label'], reference_prediction, labels=LABELS),
    index=[f'true_{LABEL_NAMES[label]}' for label in LABELS],
    columns=[f'pred_{LABEL_NAMES[label]}' for label in LABELS],
))

# Save a compact, machine-readable record of the frozen experiment.
manifest = {
    'experiment': 'Model 3: class-balanced focal XLM-R',
    'hypothesis': 'Improve mixed F1 without >0.005 macro-F1 loss versus Model 2',
    'dataset_sha256': EXPECTED_RAW_SHA256,
    'split_sha256': EXPECTED_SPLIT_SHA256,
    'model_name': MODEL_NAME,
    'beta': CB_BETA,
    'gamma': FOCAL_GAMMA,
    'class_weights': CLASS_WEIGHTS.tolist(),
    'run_seeds': RUN_SEEDS,
    'global_effective_batch_size': GLOBAL_EFFECTIVE_BATCH_SIZE,
    'test_opened': OPEN_FINAL_TEST,
}
with open(MODEL_DIR / 'experiment_manifest.json', 'w') as handle:
    json.dump(manifest, handle, indent=2)

## Final-test gate

Keep `OPEN_FINAL_TEST=False` during development. Open this gate only once, after Models 1–6, all hypotheses, hyperparameters, and the selection rule are frozen. The final table must report all three seeds, mean ± SD, per-class F1, and the mixed confusion pattern.

In [ ]:
# This cell does nothing unless the final-test gate is deliberately opened.
if not OPEN_FINAL_TEST:
    print('Final test remains sealed.')
else:
    test_dataset = SentimentDataset(test_df)
    test_rows = []

    for run_seed in RUN_SEEDS:
        state_path = MODEL_DIR / f'seed_{run_seed}' / 'best_state.pt'
        if not state_path.exists():
            raise FileNotFoundError(f'No selected state for seed {run_seed}.')

        model = build_fresh_model()
        model.load_state_dict(load_exact_state(state_path), strict=True)
        inference_args = TrainingArguments(
            output_dir=str(MODEL_DIR / f'test_seed_{run_seed}'),
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            fp16=USE_FP16, dataloader_num_workers=2, report_to=[]
        )
        inference_trainer = Trainer(
            model=model, args=inference_args,
            data_collator=data_collator, compute_metrics=compute_metrics
        )
        output = inference_trainer.predict(test_dataset)
        predictions = np.argmax(output.predictions, axis=1)
        metrics = calculate_metrics(test_df['label'].to_numpy(), predictions)
        test_rows.append({'seed': run_seed, **metrics})

        prediction_frame = test_df[['source_row_id', 'group_id', 'label']].copy()
        prediction_frame['prediction'] = predictions
        prediction_frame.to_csv(
            MODEL_DIR / f'FINAL_test_predictions_seed_{run_seed}.csv', index=False
        )
        del inference_trainer, model, output
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_results = pd.DataFrame(test_rows)
    final_summary = final_results[metric_columns].agg(['mean', 'std']).T
    final_results.to_csv(MODEL_DIR / 'FINAL_test_seed_results.csv', index=False)
    final_summary.to_csv(MODEL_DIR / 'FINAL_test_mean_std.csv')
    display(final_results.round(4))
    display(final_summary.round(4))